In [14]:
import pandas as pd
import geopandas as gpd
from thefuzz import fuzz
from thefuzz import process

# Extracting Senate and House Membership Lists

In [15]:
with open("house_list.txt", 'rt') as file:
    lines = file.readlines()

districts = [line.split(" ")[1] for line in lines]
names = [str.join(" ",line.split(" ")[5:]).split("(")[0].split("\t")[0].strip() for line in lines]
parties = []
towns = []
for line in lines:
    try:
        parties.append(line.split("(")[1].split(" ")[0])
        towns.append(line.split("-")[2].split(")")[0].strip())
    except:
        parties.append("Passamaquoddy Tribe")
        towns.append("Passamaquoddy Tribe")
representatives = pd.DataFrame.from_dict({"district": districts, "name": names, "party": parties, "town":towns})
representatives["title"] = "Rep"

with open("senate_list.txt", 'rt') as file:
    lines = file.readlines()

districts = [line.split(" ")[1] for line in lines]
names = [str.join(" ",line.split(" ")[4:]).split("(")[0].split("\t")[0].strip() for line in lines]
parties = [line.split("(")[1][0] for line in lines]
counties = [line.split("-")[2].split(")")[0].strip() for line in lines]

senators = pd.DataFrame.from_dict({"district": districts, "name": names, "party": parties, "county": counties})
senators["title"] = "Sen"

In [16]:
members = pd.concat([representatives, senators]).reset_index(drop=True)
members["first"] = members["name"].apply(lambda x: x.split(" ")[0])
members["last"] = members["name"].apply(lambda x: x.split(" ")[-1])

In [17]:
members.to_csv("member_list.csv")

# Adding Geospatial Boundaries

In [18]:
senate_districts = gpd.read_file("https://services1.arcgis.com/RbMX0mRVOFNTdLzd/arcgis/rest/services/MaineVotingDistricts2021StateSenate/FeatureServer/2/query?outFields=*&outSR=3857&where=1%3D1&f=geojson").set_index("ID")
house_districts = gpd.read_file("https://services1.arcgis.com/RbMX0mRVOFNTdLzd/arcgis/rest/services/MaineVotingDistricts2021House/FeatureServer/0/query?outFields=*&outSR=3857&where=1%3D1&f=geojson").set_index("ID")

In [19]:
senators = senate_districts.merge(senators, how='outer', left_on="DISTRICT", right_on="district")
representatives = house_districts.merge(representatives, how="outer", left_on="DISTRICT", right_on="district")

In [20]:
members_geo = pd.concat([representatives, senators]).reset_index(drop=True)
members_geo["first"] = members_geo["name"].apply(lambda x: x.split(" ")[0])
members_geo["last"] = members_geo["name"].apply(lambda x: x.split(" ")[-1])

In [21]:
members_geo.to_file('member_list.geojson')

# Extracting Committee Membership

In [22]:
def member_str_to_dict(member, committee):
    '''
    This function takes a string containing the name of the 
    committee member and other information and uses it to create 
    a dict that can then be used by pandas.
    '''
    # create the object with the committee name
    record = {"committee": committee}
    # split into two parts
    left, right = tuple(member.split(" ("))
    # extract the title (Sen or Rep) and the name from the left part
    record["title"], name = tuple(left.split(". ", maxsplit=1))
    # if the member has a second title (i.e. "Chair"), add that as a role
    if len(name.split(", ")) > 1:
        record["name"], record["role"] = tuple(name.split(", ", maxsplit=1))
    # also check if asterisk is present at the end
    elif right[-1] == '*':
        record["name"], record["role"] = name, "Ranking Minority Party Member"
        right = right[:-1]
    # otherwise, set role to None
    else:
        record["name"], record["role"] = name, None
    
    # split on hyphen to extract party and town
    if len(right[:-1].split("-", maxsplit=1)) == 2:
        record["party"], record["town"] = tuple(right[:-1].split("-", maxsplit=1))
    # catch the case where no party is listed (e.g. Passamaquoddy representative)
    else:
        record["party"], record["town"] = right[:-1], right[:-1]
    return record

In [23]:
# read the text file (copied from https://mainepolicy.org/committee-assignments-for-the-131st-legislature/)
with open("committee_list.txt", 'rt') as file:
    text = file.readlines()

# convert to a list of members
committee_members = []
committee_name = None
i = 1
while i < len(text):
    if text[i] == '\n': # indicates we are at the start of a new committee
        i += 1
        committee_name = text[i].strip()
    else:
        committee_members.append(member_str_to_dict(text[i].strip(), committee_name))
    i += 1

# turn it into a DataFrame
committee_members = pd.DataFrame.from_records(committee_members)
committee_members["first"] = committee_members["name"].apply(lambda x: x.split(" ")[0])
committee_members["last"] = committee_members["name"].apply(lambda x: x.split(" ")[-1])

# write dataframe to file
committee_members.to_csv("committee_list.csv")

In [24]:
committee_members

,committee,title,name,role,party,town,first,last
0,"Agriculture, Conservation and Forestry",Rep,William D. Pluecker,Chair,I,Warren,William,Pluecker
1,"Agriculture, Conservation and Forestry",Rep,Allison Hepler,None,D,Woolwich,Allison,Hepler
2,"Agriculture, Conservation and Forestry",Rep,Laurie Osher,None,D,Orono,Laurie,Osher
3,"Agriculture, Conservation and Forestry",Rep,Rebecca L. Jauch,None,D,Topsham,Rebecca,Jauch
4,"Agriculture, Conservation and Forestry",Rep,Kathleen A. Shaw,None,D,Auburn,Kathleen,Shaw
...,...,...,...,...,...,...,...,...
242,Joint Select Committee on Housing,Rep,Richard H. Campbell,None,R,Orrington,Richard,Campbell
243,Joint Select Committee on Housing,Rep,Mark John Blier,None,R,Buxton,Mark,Blier
244,Joint Select Committee on Housing,Sen,Teresa Pierce,Chair,D,Cumberland,Teresa,Pierce
245,Joint Select Committee on Housing,Sen,Eloise Vitelli,None,D,Sagadahoc,Eloise,Vitelli


# Fuzzy Matching Member Names
Using `thefuzz` from SeatGeek https://chairnerd.seatgeek.com/fuzzywuzzy-fuzzy-string-matching-in-python/

In [25]:
def match_member(member, matches):
    matched_name, score = process.extractOne(member, matches, scorer=fuzz.token_set_ratio)
    if score < 80:
        print("Double-check: {}\t{}".format(member, matched_name))
    return matched_name

In [26]:
committee_members["match_name"] = committee_members["name"].apply(lambda x: match_member(x, members["name"].to_list()))

Double-check: Tim Nangle	Timothy E. Nangle
Double-check: Ben Chipman	Benjamin M. Chipman
Double-check: Tim Nangle	Timothy E. Nangle


In [28]:
committee_members = gpd.GeoDataFrame(committee_members.merge(members_geo[["name", "geometry"]], left_on="match_name", right_on="name", suffixes=(None,"_y")), crs='3857')

In [29]:
committee_members.to_file('committee_list.geojson')